# Test dataset_audit.yxmd step by step

1. In **Settings**, point each Input step at a copy of its source file.
2. Run the notebook from the top. Each container's cell prints one line per step: rows and columns out, or why it failed. A failing step skips only the steps that need its rows; the rest still run.
3. `look("2.16")` shows any step's first rows and its columns (step numbers are the migration guide's).
4. `compare(...)` checks an Output step against the file the Alteryx workflow wrote.

Nothing is written while `WRITE = False`. Re-run a container's cell after changing a setting; it re-runs that container only.

In [ ]:
# Settings: edit these, then run the notebook from the top (Run All).
import pandas as pd

WORKFLOW_DIR = r"C:\path\to\workflow_folder"   # the workflow's relative paths start here
CODE_DIR = ""        # folder holding dataset_audit_generated.py and ayx_runtime.py; "" = this notebook's folder
WRITE = False        # False: Output steps keep their rows in memory (ctx.outputs) and write no file
OUTPUT_DIR = None    # with WRITE = True: write the files into this folder instead of the workflow's paths
CHAIN = True         # an Input that reads a file an earlier Output step writes gets that step's rows

# The file each Input step reads, as the workflow has it. Change a path to your own copy
# (for Excel: book.xlsx|||Sheet1$ or book.xlsx|||Sheet1$A1:F200), or give a DataFrame.
INPUTS = {
    '1': r"Input\datasets_export.csv",                          # 1.1 Input Data · container 1. Import and consolidate
    # '36': r"Output\template.xlsx|||Datasets",                   # 2.6 Dynamic Input · container 2. Dataset audit: reads the files named in [Source]; uncomment to read this one file
}
# INPUTS['30'] = pd.DataFrame({"FullPath": [...], "FileName": [...]})   # 2.1 Directory · container 2. Dataset audit: lists Output\*.xlsx

# A step that is not converted stops with NotConverted; give a function to stand in for it.
# It gets the step's input tables and returns its output table.
STUBS = {
    # '64': lambda *tables: tables[0],                            # 2.18 RegEx · container 2. Dataset audit: not converted; tool kind RegEx has no converter
}

In [ ]:
import importlib, os, sys
sys.path.insert(0, os.path.abspath(CODE_DIR or "."))
import ayx_runtime as rt
importlib.reload(rt)
wf = importlib.reload(importlib.import_module('dataset_audit_generated'))

ctx = rt.Context(workflow_dir=WORKFLOW_DIR, inputs=INPUTS, write=WRITE, out_dir=OUTPUT_DIR, stubs=STUBS)
s = {}   # every step's output so far: s[tool id]

def look(step, n=10, output=None):
    """A step's rows and columns; returns its first n rows. step: "2.16" (step number) or "44" (tool ID).
    output: "True" / "False" for a Filter, "Join" / "Left" / "Right" for a Join."""
    return rt.look(s, step, n, output, wf.STEPS, ctx)

def compare(step, legacy, sheet=None, key=None, tol=1e-6):
    """An Output step's rows against the file the legacy workflow wrote: row counts, columns, differing cells.
    Text ignores case, null and "" are alike, numbers may differ by tol. key=[columns]: line rows up by
    these columns instead of by position. Returns the differing cells as a table."""
    path = ctx.path(legacy) if isinstance(legacy, str) else legacy
    return rt.compare(rt.step_output(s, step, wf.STEPS, ctx), path, sheet, key, tol)

for k, name in wf.CONTAINERS.items():
    steps = [st for st in wf.STEPS if st[1] == k]
    print("%d. %s: %d steps (%s to %s)" % (k, name, len(steps), steps[0][2], steps[-1][2]))

## 1. Import and consolidate

Steps 1.1 to 1.4 (4 tools).

- step 1.1 reads `Input\datasets_export.csv`
- step 1.4 writes `Output\Consolidated Datasets.xlsx`, sheet Datasets

In [ ]:
s = wf.run(ctx, only=1, s=s, keep_going=True, report=True, chain=CHAIN)

## 2. Dataset audit

Steps 2.1 to 2.28 (28 tools).

- step 2.1 lists `Output` (`*.xlsx`)
- step 2.6 reads files like `Output\template.xlsx`, sheet Datasets
- step 2.28 writes `Output\Dataset audit.xlsx`, sheet Issues

In [ ]:
s = wf.run(ctx, only=2, s=s, keep_going=True, report=True, chain=CHAIN)

## 3. Old checks

Disabled in the workflow: nothing to run.

## Steps without an output

Every step that failed, and the steps skipped because they need its rows.

In [ ]:
rt.problems(ctx, wf.STEPS)

## Look at a step

Any step number from the guide (or a tool ID). For a Filter, `look("2.2", output="False")` shows the rows that failed the test.

In [ ]:
look("1.3")

## Compare with the legacy output

Each line compares one Output step with a file the Alteryx workflow wrote. The paths are the workflow's own; point them at a copy of the legacy output. Add `key=["ID"]` to line rows up by key columns instead of by position.

In [ ]:
diff_1_4 = compare("1.4", r"Output\Consolidated Datasets.xlsx", 'Datasets')  # 1.4 Output Data · container 1. Import and consolidate
diff_2_28 = compare("2.28", r"Output\Dataset audit.xlsx", 'Issues')     # 2.28 Output Data · container 2. Dataset audit